# Second case: the tier's question and Marketing's pushback, argued from the same numbers

**Week 1, Tuesday afternoon. The second case, in pairs, forty minutes.** One of you answers Marketing,
the other answers the head of Retail-Plus, from the same file, and together you write the reply.

> **Marketing comes back with a new deck.** "Student orders per customer rose 40 percent after our
> campus push, so acquisition works. And Retail-Plus web orders fell hardest, 24 to 9, so this is the
> website team's problem, not the tier's and not the app's."
>
> The marketing lead, Kalpa Retail

> **The tier asks.** "Fine. Which of my members do I call first?"
>
> The head of Retail-Plus

> **Kavya's review of the escalated case.** "You found a branch that moved for a reason nobody
> guessed. Do the same with Marketing's two new numbers: find what each is made of before you agree
> or disagree."

This is the exercise twin: each placeholder is a lettered choice in the comment above it. Run it from the top; it stops at the first placeholder you have not filled, which is intended.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (chapter 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers if customers else 0,
            "revenue_per_order": revenue / orders if orders else 0}


def pct_change(before, after):
    """Chapter 5's fixed helper: the change every time, in the same type."""
    return round(100 * (after - before) / before, 1)

print(len(ORDERS), "orders loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["Student +40 percent: acquisition?", "Web fell hardest: the website?",
                "Who to call first", "The reply and the evidence"], show=False),
    kit.matrix(["Marketing", "the head of Retail-Plus"], ["claim", "test"],
               [["campaigns work; the website", "new ids; web in another segment"], ["which members", "orders per member, Q1 to Q2"]],
               show=False),
)

## Part 1. "Student orders per customer rose 40 percent, so acquisition works"

Acquisition means new customers. Count the Student ids in each quarter and the ones that are new.

In [ ]:
s_ids = {q: {o["customer_id"] for o in ORDERS if o["segment"] == "Student" and o["quarter"] == q} for q in ("Q1", "Q2")}
s_orders = {q: sum(1 for o in ORDERS if o["segment"] == "Student" and o["quarter"] == q) for q in ("Q1", "Q2")}
# TODO 1. Which set holds the Student customers who are new in Q2?
#   a) s_ids["Q2"] - s_ids["Q1"]
#   b) s_ids["Q1"] - s_ids["Q2"]
#   c) s_ids["Q1"] & s_ids["Q2"]
#   d) len(s_ids["Q2"]) - len(s_ids["Q1"])
new_students = __TODO1__
kit.columns(["Q1", "Q2"], [("Student customers", [len(s_ids["Q1"]), len(s_ids["Q2"])]), ("Student orders", [s_orders["Q1"], s_orders["Q2"]])],
            width=520, title="Student: the same two customers, two more orders")
kit.check("no Student customer is new in Q2", len(new_students) == 0)
kit.check("the 40 percent is 5 orders becoming 7 on 2 customers", (s_orders["Q1"], s_orders["Q2"], len(s_ids["Q2"])) == (5, 7, 2))

## Part 2. "Retail-Plus web orders fell hardest, so it is the website"

A broken website hurts every customer who uses it. If the website were the cause, what would another
segment's web orders show?

In [ ]:
web = {(o["segment"], o["quarter"]): 0 for o in ORDERS}
for o in ORDERS:
    if o["channel"] == "web":
        web[(o["segment"], o["quarter"])] += 1
# TODO 2. If the website were broken for everyone, which other segment's web orders would also have fallen hard?
#   a) "Retail-Plus"
#   b) "Student"
#   c) "Retail-Core"
#   d) "none"
also_falls = __TODO2__
kit.columns(["Retail-Plus", "Retail-Core"], [("Q1 web orders", [web[("Retail-Plus", "Q1")], web[("Retail-Core", "Q1")]]),
                                             ("Q2 web orders", [web[("Retail-Plus", "Q2")], web[("Retail-Core", "Q2")]])],
            width=560, title="Web orders by segment: members fell, Retail-Core held on the same website")
kit.check("the test names the comparison segment", also_falls == "Retail-Core")
kit.check("Retail-Core's web orders held, 13 to 12", (web[("Retail-Core", "Q1")], web[("Retail-Core", "Q2")]) == (13, 12))
kit.check("Retail-Plus web orders fell 24 to 9", (web[("Retail-Plus", "Q1")], web[("Retail-Plus", "Q2")]) == (24, 9))

## Part 3. "Which of my members do I call first?"

Count each member's orders in Q1 and Q2, and put the members who slowed most at the top of the list.

In [ ]:
per = {q: {} for q in ("Q1", "Q2")}
for o in ORDERS:
    if o["segment"] == "Retail-Plus":
        per[o["quarter"]][o["customer_id"]] = per[o["quarter"]].get(o["customer_id"], 0) + 1
pairs = {}
for cid in per["Q1"]:
    key = (per["Q1"][cid], per["Q2"].get(cid, 0))
    pairs[key] = pairs.get(key, 0) + 1
call_first = []
for cid in per["Q1"]:
    q1n, q2n = per["Q1"][cid], per["Q2"].get(cid, 0)
    # TODO 3. Which members go to the top of the call list?
    #   a) q2n == 0
    #   b) q1n - q2n >= 2
    #   c) q2n > q1n
    #   d) q1n == 1
    if __TODO3__:
        call_first.append(cid)
labels = [f"{a} to {b}" for a, b in sorted(pairs, reverse=True)]
kit.bars([(l, pairs[k]) for l, k in zip(labels, sorted(pairs, reverse=True))], lit=[0],
         title="Retail-Plus members by orders in Q1 to orders in Q2")
kit.check("7 members fell from three orders to one", len(call_first) == 7 and pairs[(3, 1)] == 7)
kit.check("every member still ordered in Q2", all(per["Q2"].get(c, 0) > 0 for c in per["Q1"]))

## Part 4. The reply, and the one request that settles most

In [ ]:
EVIDENCE = {
    "campaigns": "Marketing's campaign reach by month",
    "export": "this export, cut another way",
    "app_logs": "the app's reorder attempts and failures by week, and the release date",
    "tier_log": "the tier's change log for July, renewals and support tickets",
}
# TODO 4. The fall began in July and the button broke on 25 August. Which request can settle the larger part of the fall?
#   a) "campaigns"
#   b) "export"
#   c) "app_logs"
#   d) "tier_log"
first_request = __TODO4__
kit.tree({"label": "the reply", "kind": "lit", "branches": [
    ("to Marketing", {"label": f"Student: {len(new_students)} new ids\nweb tested on {also_falls}", "kind": "known"}),
    ("to the tier", {"label": f"call the {len(call_first)} who fell\nby two or more orders", "kind": "known"}),
    ("first request", {"label": EVIDENCE[first_request][:34], "kind": "unknown"})]},
    title="The pair's reply, in three branches")
kit.check("the first request targets the July change, which predates the button", first_request == "tier_log")
kit.check("the request is for data this export does not carry", first_request not in ("export", "campaigns"))

In [ ]:
kit.check_summary()